In [ ]:
!pip install -q kagglehub

import kagglehub
kagglehub.login()

In [ ]:
path = kagglehub.dataset_download("jessicali9530/kuc-hackathon-winter-2018")
print("Downloaded to:", path)

import os
tsv_candidates = [os.path.join(root, f) for root, _, files in os.walk(path) for f in files if f.endswith(".tsv")]
print("TSV files found:", tsv_candidates)

In [ ]:
import pandas as pd

train_path = next(p for p in tsv_candidates if "Train" in p)
test_path = next(p for p in tsv_candidates if "Test" in p)

train_df = pd.read_csv(train_path, sep="\t")
test_df = pd.read_csv(test_path, sep="\t")

df = pd.concat([train_df, test_df], ignore_index=True)
print("Combined shape:", df.shape)
print("Null conditions:", df["condition"].isna().sum())

df = df.dropna(subset=["condition", "drugName"]).reset_index(drop=True)
print("After dropping nulls:", df.shape)

In [ ]:
df["condition_len"] = df["condition"].str.len()

print("Condition length distribution:")
print(df["condition_len"].describe())
print()
print("Longest condition strings (inspect these for garbled/junk data):")
print(df.drop_duplicates(subset="condition").nlargest(10, "condition_len")[["condition", "condition_len"]].to_string())

In [ ]:
JUNK_LENGTH_THRESHOLD = 100  # adjust based on cell 4's actual output

junk_mask = df["condition_len"] > JUNK_LENGTH_THRESHOLD
print(f"Rows identified as junk: {junk_mask.sum()} ({junk_mask.mean():.2%})")

df = df[~junk_mask].drop(columns=["condition_len"]).reset_index(drop=True)
print("Final shape:", df.shape)
print("Unique conditions:", df["condition"].nunique())

In [ ]:
df["date"] = pd.to_datetime(df["date"], format="%d-%b-%Y")
df["year"] = df["date"].dt.year
df["alpha"] = df["year"] / df.groupby("condition")["year"].transform("max")  # recency weight

per_drug = df.groupby(["condition", "drugName"]).apply(
    lambda g: pd.Series({
        "avg_rating": g["rating"].mean(),
        "avg_alpha": g["alpha"].mean(),
        "total_useful": g["usefulCount"].sum(),
        "review_count": len(g),
    })
).reset_index()

per_drug["max_useful_in_condition"] = per_drug.groupby("condition")["total_useful"].transform("max")
per_drug["beta"] = (per_drug["total_useful"] / per_drug["max_useful_in_condition"]).fillna(0)

raw_score = per_drug["avg_rating"] * per_drug["avg_alpha"] + per_drug["beta"]
per_drug["score"] = (raw_score / raw_score.max() * 10).round(2)

print(per_drug.sort_values("score", ascending=False).head(10))

In [ ]:
MIN_REVIEWS = 5

before = len(per_drug)
per_drug = per_drug[per_drug["review_count"] >= MIN_REVIEWS].reset_index(drop=True)
print(f"Filtered out {before - len(per_drug)} (condition, drug) pairs with fewer than {MIN_REVIEWS} reviews")
print(f"Remaining: {len(per_drug)} pairs across {per_drug['condition'].nunique()} conditions")

In [ ]:
TOP_K_PER_CONDITION = 15

rankings = {}
for condition, group in per_drug.groupby("condition"):
    top = group.sort_values("score", ascending=False).head(TOP_K_PER_CONDITION)
    rankings[condition] = [
        {"drug": row["drugName"], "score": float(row["score"]), "review_count": int(row["review_count"])}
        for _, row in top.iterrows()
    ]

print(f"{len(rankings)} conditions ready for export")
# Spot check one
example_condition = list(rankings.keys())[0]
print(f"\nExample — {example_condition}:")
for entry in rankings[example_condition][:5]:
    print(" ", entry)

In [ ]:
import json, os

os.makedirs("data_artifacts", exist_ok=True)
with open("data_artifacts/drug_rankings.json", "w") as f:
    json.dump(rankings, f, indent=2)

print("Exported drug_rankings.json")
!ls -la data_artifacts

In [ ]:
contraindication_rules = {
    "allergy_rules": {
        # Well-known cross-reactivity classes — illustrative only, NOT exhaustive.
        "penicillin": ["amoxicillin", "ampicillin", "penicillin v potassium"],
        "sulfa": ["sulfamethoxazole", "sulfasalazine"],
        "nsaid": ["ibuprofen", "naproxen", "aspirin"],
    },
    "interaction_rules": {
        # Well-known, commonly-cited interaction pairs — illustrative only.
        "warfarin": ["ibuprofen", "aspirin", "naproxen"],
        "maoi": ["sertraline", "fluoxetine"],
    },
}

with open("data_artifacts/contraindication_rules.json", "w") as f:
    json.dump(contraindication_rules, f, indent=2)

print("Exported contraindication_rules.json (illustrative only — see service README)")

In [ ]:
import shutil
shutil.make_archive("drug_recommendation_artifacts", "zip", "data_artifacts")

from google.colab import files
files.download("drug_recommendation_artifacts.zip")